# Fase 5 — experimentos en GPU (Kaggle)

Corre los experimentos de la Fase 5 de `analyst-in-the-middle` de principio a fin, sin
intervención: instala Ollama, descarga los modelos, reconstruye el dataset y los reportes
envenenados (idénticos al manifiesto versionado), corre E1 → E3 → E4 (y E2 si se pide),
genera tablas y gráficas con `aitm report` y deja todo en un zip en la salida.

**Cómo correrlo sin dejar la pestaña abierta (Kaggle):**

1. *Settings* → **Accelerator: GPU T4 x2** (o P100) y **Internet: On** (requiere cuenta verificada con teléfono).
2. **Save Version → Save & Run All (Commit)**. Puedes cerrar el navegador.
3. Al terminar, en la versión guardada → *Output* → descarga `resultados_fase5.zip`.

Tiempo esperado: ~1,5 h el núcleo (E1+E3+E4), ~2,5 h con E2. Kaggle permite hasta 12 h por corrida.

> En Google Colab también funciona celda por celda, pero la versión gratuita **se detiene
> si se cierra la pestaña**. Para correr desatendido, usar Kaggle.

**Nota de alcance:** los reportes envenenados son material de laboratorio generado desde
un corpus público (AnnoCTR) y se procesan en una VM efímera con modelos locales (Ollama);
no se envían a ninguna API de terceros. Ver `docs/ARCHITECTURE.md`, sección 7.

In [ ]:
# --- Parámetros -------------------------------------------------------------
REPO = "https://github.com/marianamalagon11/analyst-in-the-middle.git"
BRANCH = "feature/phase4"       # rama con el código de la Fase 5
RUN_E2 = False                  # comparación de modelos (mistral y qwen2.5): ~1 h más
CORE = ["e1_vectores", "e3_defensas", "e4_combinadas"]   # en este orden: aprovecha la cache

import os, pathlib, shutil, subprocess, time
OUT = pathlib.Path("/kaggle/working") if pathlib.Path("/kaggle/working").exists() else pathlib.Path("/content")
WORK = pathlib.Path("/tmp/aitm")
print("salida:", OUT)
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Ollama y modelos

In [ ]:
!apt-get -qq update >/dev/null && apt-get -qq install -y pciutils lshw zstd >/dev/null
!curl -fsSL https://ollama.com/install.sh | sh

env = dict(os.environ, OLLAMA_MAX_LOADED_MODELS="2", OLLAMA_KEEP_ALIVE="30m")
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/tmp/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    if subprocess.run(["ollama", "list"], capture_output=True).returncode == 0:
        break
    time.sleep(1)
!ollama --version

In [ ]:
models = ["llama3.1:8b-instruct-q4_K_M", "qwen2.5:7b-instruct"] + (["mistral:7b-instruct"] if RUN_E2 else [])
for m in models:
    subprocess.run(["ollama", "pull", m], check=True, stdout=subprocess.DEVNULL)
!ollama list

## 2. Código, dataset y reportes envenenados

In [ ]:
if WORK.exists():
    shutil.rmtree(WORK)
!git clone -q --branch {BRANCH} {REPO} {WORK}
os.chdir(WORK)
!pip install -q -e ".[visual]"
!git log --oneline -1

In [ ]:
# El corpus y los documentos no se versionan: se reconstruyen de forma determinista.
!python scripts/fetch_dataset.py | tail -2
!python scripts/build_dataset.py --force --pin lab52_2021-05-17_literature-lover-targeting-colombia-with | tail -2
!git checkout -q data/processed/dataset.json
!aitm attack | tail -4
# Debe ser byte a byte el manifiesto versionado (Fase 3):
r = subprocess.run(["git", "diff", "--quiet", "data/poisoned/manifest.jsonl"])
print("manifiesto idéntico al versionado:", r.returncode == 0)
assert r.returncode == 0, "los reportes envenenados no coinciden con el manifiesto: no seguir"

## 3. Experimentos

Cada experimento imprime primero su estimado (`--dry-run`) y luego corre. Al terminar cada uno se copia su carpeta a la salida, para no perder nada si la sesión se corta.

In [ ]:
def checkpoint():
    dest = OUT / "runs"
    for d in pathlib.Path("results/runs").glob("e*"):
        shutil.copytree(d, dest / d.name, dirs_exist_ok=True)

experiments = CORE + (["e2_modelos"] if RUN_E2 else [])
for name in experiments:
    print(f"\n===== {name} =====")
    !aitm run experiments/{name}.yaml --dry-run
    started = time.time()
    !aitm run experiments/{name}.yaml
    print(f"{name}: {(time.time() - started) / 60:.1f} min")
    checkpoint()

## 4. Tablas, gráficas y zip

In [ ]:
!aitm report
for t in sorted(pathlib.Path("results/tables").glob("exp*.md")):
    print("\n" + t.read_text())

In [ ]:
bundle = WORK / "resultados_fase5"
if bundle.exists():
    shutil.rmtree(bundle)
for d in pathlib.Path("results/runs").glob("e*"):
    shutil.copytree(d, bundle / "results/runs" / d.name)
for pattern in ["results/tables/exp*", "results/tables/asr_celdas.csv", "results/figures/*.png"]:
    for f in pathlib.Path(".").glob(pattern):
        (bundle / f.parent).mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, bundle / f)
shutil.copy2("/tmp/ollama.log", bundle / "ollama.log")
shutil.make_archive(str(OUT / "resultados_fase5"), "zip", bundle)
print("listo:", OUT / "resultados_fase5.zip")
server.terminate()